# Week 7: Integrated System Design

In the previous six weeks you learned individual skills: writing clean Python, testing and debugging, performance tuning, software design,building a web API, and Cloud Deployment & DevOps. This week is about connecting those skills into one working system.

We will use a small, everyday example: **an online bookstore that takes orders and checks stock**. It is *simple on purpose*, so the focus stays on how the pieces fit together, not on the domain.

By the end of this notebook you will have:

1. Built a small data pipeline
2. Split the system into two independent microservices
3. Measured a basic performance trade-off
4. Added monitoring and logging
5. Added basic security checks
6. Run the whole system end to end


## 1. Designing a scalable data pipeline

A data pipeline takes raw input, processes it in stages, and produces clean output.

 Two common styles are:
- **Batch processing**: collect a group of items, then process them all at once.
- **Streaming (one at a time) processing**: process each item as soon as it arrives.

> Below we simulate a list of incoming book orders and process them both ways, so you can see the difference in code shape.


In [3]:
raw_orders = [
    {"book_id": "book_001", "quantity": 2},
    {"book_id": "book_002", "quantity": 1},
    {"book_id": "book_003", "quantity": 5},
    {"book_id": "book_001", "quantity": 1},
]

def clean_order(order):
    """Add a total_items field and make sure quantity is a positive int."""
    order = dict(order)
    order["quantity"] = max(1, int(order["quantity"]))
    return order


# --- Batch style: process the whole list at once ---
def process_batch(orders):
    return [clean_order(o) for o in orders]

batch_result = process_batch(raw_orders)
batch_result

[{'book_id': 'book_001', 'quantity': 2},
 {'book_id': 'book_002', 'quantity': 1},
 {'book_id': 'book_003', 'quantity': 5},
 {'book_id': 'book_001', 'quantity': 1}]

In [4]:
# --- Streaming style: process one order at a time as it "arrives" ---
def order_stream(orders):
    for order in orders:
        yield clean_order(order)

for cleaned in order_stream(raw_orders):
    print(cleaned)

{'book_id': 'book_001', 'quantity': 2}
{'book_id': 'book_002', 'quantity': 1}
{'book_id': 'book_003', 'quantity': 5}
{'book_id': 'book_001', 'quantity': 1}


**Why this matters:**
- **Batch processing** is simpler and works well for small datasets, but it waits until the entire batch has been processed before returning the results. Batch processing is suitable when all the data are already available and can comfortably fit in memory, providing a simple approach for processing the entire dataset at once. 
- **Streaming** processes each item as it arrives, allowing immediate actions, such as updating a bookstore's inventory after each order, while using much less memory for large datasets. Streaming with `yield` is preferable when working with large datasets, files, API responses, sensor data, or message queues, where processing one item at a time reduces memory usage and allows results to be handled immediately. The trade-off is slightly more complex code to manage the stream.
---


## 2. Microservices architecture

Instead of one large program, we will split the bookstore into two small, independent services:

- [**Order Service**](order_service.py) (port 5000): receives orders, checks them for mistakes, and asks the Inventory Service if the book is available.
- [**Inventory Service**](inventory_service.py) (port 5001): keeps the stock numbers and answers stock checks.

They only talk to each other over HTTP, the same way they would talk to any other service on the internet. This means each one can be built, tested, and even scaled separately.

Run the cell below to write both services to disk.


In [1]:
%%writefile order_service.py 
# Jupyter Notebook magic command: Writes all code in this cell into a Python file named order_service.py
# This allows us to create a standalone microservice file from the notebook.
"""
Order Service
This microservice receives book orders, validates incoming requests, and communicates with the Inventory Service to check product availability.
The example demonstrates key microservice concepts, including API endpoints, service-to-service communication, logging, security checks, and monitoring.
"""
from flask import Flask, request, jsonify
import requests
import logging
import time
import uuid
import os

app = Flask(__name__)

# Configure structured logging for monitoring.
logging.basicConfig(
    level=logging.INFO,
    format='{"time":"%(asctime)s","service":"order","level":"%(levelname)s","message":"%(message)s"}'
)
logger = logging.getLogger("order_service")

# Inventory service URL from environment variable.
INVENTORY_SERVICE_URL = os.environ.get("INVENTORY_SERVICE_URL", "http://localhost:5001")

# Demo API key for request authentication.
API_KEY = "week7-secret-key"

# Simple application metrics.
metrics = {
    "requests_received": 0,
    "orders_placed": 0,
    "orders_rejected": 0,
    "total_response_time": 0.0,
}


def require_api_key(req):
    """Validate the API key in the request header."""
    return req.headers.get("X-API-Key") == API_KEY


@app.route("/order", methods=["POST"])
def place_order():
    start_time = time.time()
    metrics["requests_received"] += 1

    # Generate unique ID for request tracing.
    request_id = str(uuid.uuid4())[:8]

    # Check request authentication.
    if not require_api_key(request):
        logger.warning(f"[{request_id}] invalid API key")
        metrics["orders_rejected"] += 1
        return jsonify({"error": "invalid API key"}), 401

    # Read request JSON payload.
    data = request.get_json(silent=True)

    # Validate required fields.
    if not data or "book_id" not in data or "quantity" not in data:
        logger.warning(f"[{request_id}] invalid payload")
        metrics["orders_rejected"] += 1
        return jsonify({"error": "order must include book_id and quantity"}), 400

    # Validate order quantity.
    if not isinstance(data["quantity"], int) or data["quantity"] <= 0:
        logger.warning(f"[{request_id}] invalid quantity")
        metrics["orders_rejected"] += 1
        return jsonify({"error": "quantity must be a positive integer"}), 400

    logger.info(f"[{request_id}] processing order")

    # Request stock information from inventory service.
    try:
        response = requests.post(
            f"{INVENTORY_SERVICE_URL}/check_stock",
            json={
                "book_id": data["book_id"],
                "quantity": data["quantity"]
            },
            timeout=3,
        )
        stock_result = response.json()

    except requests.exceptions.RequestException as e:
        logger.error(f"[{request_id}] inventory unavailable: {e}")
        return jsonify({"error": "inventory service unavailable"}), 503

    # Measure response time.
    elapsed = time.time() - start_time
    metrics["total_response_time"] += elapsed

    # Confirm order if stock is available.
    if stock_result.get("in_stock"):
        metrics["orders_placed"] += 1
        logger.info(f"[{request_id}] order confirmed")

        return jsonify({
            "request_id": request_id,
            "status": "confirmed",
            "book_id": data["book_id"],
            "quantity": data["quantity"],
        }), 201

    # Reject order if stock is unavailable.
    metrics["orders_rejected"] += 1
    logger.info(f"[{request_id}] out of stock")

    return jsonify({
        "request_id": request_id,
        "status": "out_of_stock",
        "book_id": data["book_id"],
    }), 409


@app.route("/health", methods=["GET"])
def health():
    # Health check endpoint for monitoring.
    return jsonify({"status": "ok", "service": "order_service"}), 200


@app.route("/metrics", methods=["GET"])
def get_metrics():
    # Calculate average successful request time.
    avg_time = (
        metrics["total_response_time"] / metrics["orders_placed"]
        if metrics["orders_placed"] > 0 else 0
    )

    return jsonify({
        "requests_received": metrics["requests_received"],
        "orders_placed": metrics["orders_placed"],
        "orders_rejected": metrics["orders_rejected"],
        "average_response_time_seconds": round(avg_time, 4),
    }), 200


if __name__ == "__main__":
    # Start Flask service.
    app.run(host="0.0.0.0", port=5000, debug=False)



Overwriting order_service.py


In [2]:
%%writefile inventory_service.py
from flask import Flask, request, jsonify
import logging

# Create the Flask application.
app = Flask(__name__)


# Configure structured logging for monitoring and debugging.
logging.basicConfig(
    level=logging.INFO,
    format='{"time":"%(asctime)s","service":"inventory","level":"%(levelname)s","message":"%(message)s"}'
)

# Create a logger for this microservice.
logger = logging.getLogger("inventory_service")


# Simulated inventory database.
# In a real system, this data would usually be stored in a database.
stock = {
    "book_001": 5,
    "book_002": 0,
    "book_003": 12,
}


@app.route("/check_stock", methods=["POST"])
def check_stock():
    # Receive the order information sent by the Order Service.
    data = request.get_json(silent=True)

    # Validate that the request contains the required fields.
    if not data or "book_id" not in data or "quantity" not in data:
        logger.warning("rejected malformed stock check request")
        return jsonify({"error": "book_id and quantity are required"}), 400

    # Extract book identifier and requested quantity.
    book_id = data["book_id"]
    quantity = data["quantity"]

    # Retrieve the current stock level.
    # Return 0 if the requested book does not exist.
    available = stock.get(book_id, 0)


    # Check whether enough items are available.
    if available >= quantity:

        # Reduce the inventory after approving the order.
        stock[book_id] -= quantity

        # Log successful stock reservation.
        logger.info(
            f"book_id={book_id} qty={quantity} approved, "
            f"{stock[book_id]} left"
        )

        # Return a successful stock confirmation response.
        return jsonify({
            "in_stock": True,
            "remaining": stock[book_id]
        }), 200


    else:
        # Log failed stock validation.
        logger.info(
            f"book_id={book_id} qty={quantity} denied, "
            f"only {available} left"
        )

        # Inform the Order Service that the item is unavailable.
        return jsonify({
            "in_stock": False,
            "remaining": available
        }), 200



@app.route("/health", methods=["GET"])
def health():
    # Health endpoint used by monitoring tools and Kubernetes probes.
    return jsonify({
        "status": "ok",
        "service": "inventory_service"
    }), 200



if __name__ == "__main__":
    # Start the Inventory Service on port 5001.
    app.run(host="0.0.0.0", port=5001, debug=False)

Overwriting inventory_service.py


### Running the services with Docker

So far both services run as plain Python processes. You can also package each one into its own Docker image and start both together with Docker Compose. This mirrors what you did with the Dockerfile in Weeks 5-6, **now with two containers that need to find each other over the network**.



In [3]:
%%writefile Dockerfile.order
# Use a lightweight Python 3.11 base image.
FROM python:3.11-slim

# Set the working directory inside the container.
WORKDIR /app

# Copy dependency file into the container.
COPY requirements.txt .

# Install required Python packages.
# --no-cache-dir reduces the image size by removing pip cache files.
RUN pip install --no-cache-dir -r requirements.txt

# Copy the application code into the container.
COPY order_service.py .

# Document that the application listens on port 5000.
EXPOSE 5000

# Start the Order Service when the container runs.
CMD ["python", "order_service.py"]

Overwriting Dockerfile.order


In [4]:
%%writefile Dockerfile.inventory
# Use a lightweight Python 3.11 image as the base environment.
FROM python:3.11-slim

# Set the working directory inside the container.
WORKDIR /app

# Copy the dependency file into the container.
COPY requirements.txt .

# Install Python dependencies required by the service.
# The cache is disabled to keep the image smaller.
RUN pip install --no-cache-dir -r requirements.txt

# Copy the Inventory Service application code.
COPY inventory_service.py .

# Document the port used by the Flask application.
EXPOSE 5001

# Start the Inventory Service when the container launches.
CMD ["python", "inventory_service.py"]


Overwriting Dockerfile.inventory


In [5]:
%%writefile docker-compose.yml
# Define the services that make up the application.
services:

  # Inventory microservice.
  inventory:
    # Build the image using the specified Dockerfile.
    build:
      context: .
      dockerfile: Dockerfile.inventory

    # Map container port 5001 to the host machine.
    ports:
      - "5001:5001"


  # Order microservice.
  order:
    # Build the image using the specified Dockerfile.
    build:
      context: .
      dockerfile: Dockerfile.order

    # Expose the Order Service on port 5000.
    ports:
      - "5000:5000"

    # Start the Inventory Service before starting the Order Service.
    depends_on:
      - inventory

    # Configure the Inventory Service URL inside the container.
    # Docker Compose provides service-name-based networking.
    # "inventory" resolves automatically to the inventory container.
    environment:
      - INVENTORY_SERVICE_URL=http://inventory:5001


Overwriting docker-compose.yml


### Note: 
- The  [`docker-compose.yml`](docker-compose.yml) sets `INVENTORY_SERVICE_URL=http://inventory:5001` for the order service, instead of `localhost`. 
- Inside Docker Compose, each service can reach another by its service name (`inventory`), because Compose puts them on the same private network. 
- [`order_service.py`](order_service.py) reads this from an environment variable, falling back to `localhost` when it isn't set, so the same code works whether you run it directly or inside a container.

> To build and start both containers, run this in a terminal, in the same folder as the files above:

```bash
docker compose up --build # Build the Docker images and start all services.
```

Then, in a separate terminal or in this notebook, you can hit the same endpoints as before:


> Test the health endpoints
Open these in your browser:
1. **Inventory Service**: http://localhost:5001/health
2. **Order Service**: http://localhost:5000/health

In [6]:
# **Only run this after `docker compose up --build` is running in a terminal**
import requests
print(requests.get("http://localhost:5001/health").json())
print(requests.get("http://localhost:5000/health").json())

{'service': 'inventory_service', 'status': 'ok'}
{'service': 'order_service', 'status': 'ok'}


### Next: Test the microservice communication

> Keep docker compose up running 



In [7]:
import requests

response = requests.post(
    "http://localhost:5000/order",
    headers={"X-API-Key": "week7-secret-key"},
    json={
        "book_id": "book_001",
        "quantity": 2
    }
)

print(response.json())

{'book_id': 'book_001', 'quantity': 2, 'request_id': 'af5fee39', 'status': 'confirmed'}


**Complete Request Flow:**

```Client (Python requests)
          |
          v
Order Service (localhost:5000)
          |
          |  HTTP request inside Docker network
          v
Inventory Service (inventory:5001)
          |
          v
Stock updated successfully
```

When you're done, stop the containers with `docker compose down` in the terminal where they're running.

### Starting both services

Normally you would run each service in its own terminal window with `python order_service.py` and `python inventory_service.py`. To keep this notebook self-contained, the cell below starts both as background processes instead.


In [8]:
import subprocess
import time
import requests

inventory_process = subprocess.Popen(["python3", "inventory_service.py"])
time.sleep(1.5)
order_process = subprocess.Popen(["python3", "order_service.py"])
time.sleep(1.5)

print(requests.get("http://localhost:5001/health").json())
print(requests.get("http://localhost:5000/health").json())

{'service': 'inventory_service', 'status': 'ok'}
{'service': 'order_service', 'status': 'ok'}


## 3. Performance trade-offs

Now that both services are running, let's compare two ways of sending orders to the Order Service: **one at a time (sequential)** versus **several at once (concurrent)**.


In [9]:
import concurrent.futures
import requests
import time

# Authentication key for accessing the Order Service.
API_KEY = "week7-secret-key"

# Order Service API endpoint.
ORDER_URL = "http://localhost:5000/order"

# Generate multiple test orders for benchmarking.
test_orders = [{"book_id": "book_003", "quantity": 1} for _ in range(10)]


def send_order(order):
    # Send one order request and return the HTTP status code.
    return requests.post(
        ORDER_URL,
        json=order,
        headers={"X-API-Key": API_KEY}
    ).status_code


# Measure execution time for sequential requests.
start = time.time()

for order in test_orders:
    send_order(order)

sequential_time = time.time() - start


# Measure execution time for concurrent requests.
start = time.time()

# Create a pool of worker threads to send requests in parallel.
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    list(executor.map(send_order, test_orders))

concurrent_time = time.time() - start


# Compare the performance of both approaches.
print(f"Sequential: {sequential_time:.3f}s")
print(f"Concurrent: {concurrent_time:.3f}s")

Sequential: 40.752s
Concurrent: 8.212s


**What this shows:** 
- Sending requests concurrently usually finishes faster because the **Order Service** can start working on a new request while it is still waiting on the network for another one to reach the Inventory Service. 
- This is a throughput gain, not free computation. Each request still costs the same amount of work; we are simply overlapping the waiting time. 
- A real system has to balance this against how many concurrent requests a service (and its database) can safely handle at once.

## 4. Monitoring and logging

Both services write structured log lines (you should see them appear in the terminal or notebook kernel output as JSON). The Order Service also exposes a `/metrics` endpoint, which is a common pattern for letting monitoring tools check on a service without reading its logs directly.


In [10]:
print(requests.get("http://localhost:5000/metrics").json())

{'average_response_time_seconds': 3.4073, 'orders_placed': 12, 'orders_rejected': 8, 'requests_received': 20}


**Why this matters:**
- In a system with many services, you cannot watch a single terminal to know what is happening. 
- Structured logs (consistent fields like `time`, `service`, `level`, `message`) can be collected and searched automatically. 
- A `/metrics` endpoint lets a monitoring tool poll each service on a schedule and alert someone if, for example, `orders_rejected` suddenly spikes.

## 5. Security considerations

The Order Service checks two things before accepting an order:

1. An API key, sent in the `X-API-Key` header
2. That the order body actually contains a valid `book_id` and a positive `quantity`

Let's confirm both checks work.


In [11]:
# Missing API key
r1 = requests.post(ORDER_URL, json={"book_id": "book_001", "quantity": 1})
print("No API key:", r1.status_code, r1.json())

# Malformed order (missing quantity)
r2 = requests.post(ORDER_URL, json={"book_id": "book_001"}, headers={"X-API-Key": API_KEY})
print("Malformed order:", r2.status_code, r2.json())

# Negative quantity
r3 = requests.post(ORDER_URL, json={"book_id": "book_001", "quantity": -3}, headers={"X-API-Key": API_KEY})
print("Negative quantity:", r3.status_code, r3.json())

No API key: 401 {'error': 'invalid API key'}
Malformed order: 400 {'error': 'order must include book_id and quantity'}
Negative quantity: 400 {'error': 'quantity must be a positive integer'}


**What is missing for a real production system:**

- The API key here is a single hardcoded string. A real system would issue a separate key per client and store keys securely, not in the source code.
- All traffic here runs over plain HTTP. Production traffic should run over HTTPS so requests cannot be read or altered in transit.
- There is no rate limiting yet, so one client could send unlimited requests. You will add a simple version of this in the extension exercises.
- Error messages should avoid leaking internal details (for example, stack traces) to the client.


## 6. Connecting it all together

This final cell places several orders through the running system and prints the results, tying together the pipeline, the two services, the logging, and the metrics endpoint.


In [12]:
sample_orders = [
    {"book_id": "book_001", "quantity": 2},
    {"book_id": "book_002", "quantity": 1},   # out of stock
    {"book_id": "book_003", "quantity": 20},  # more than available
    {"book_id": "book_001", "quantity": 1},
]

for order in sample_orders:
    r = requests.post(ORDER_URL, json=order, headers={"X-API-Key": API_KEY})
    print(order, "->", r.status_code, r.json())

print("\nFinal metrics:")
print(requests.get("http://localhost:5000/metrics").json())

{'book_id': 'book_001', 'quantity': 2} -> 201 {'book_id': 'book_001', 'quantity': 2, 'request_id': '0248876b', 'status': 'confirmed'}
{'book_id': 'book_002', 'quantity': 1} -> 409 {'book_id': 'book_002', 'request_id': '636c5304', 'status': 'out_of_stock'}
{'book_id': 'book_003', 'quantity': 20} -> 409 {'book_id': 'book_003', 'request_id': 'ef33e05c', 'status': 'out_of_stock'}
{'book_id': 'book_001', 'quantity': 1} -> 201 {'book_id': 'book_001', 'quantity': 1, 'request_id': '0c014cdf', 'status': 'confirmed'}

Final metrics:
{'average_response_time_seconds': 3.5067, 'orders_placed': 14, 'orders_rejected': 13, 'requests_received': 27}


### Shutting down

Run this cell when you are done experimenting (**AFTER** solving the exercises), to stop both background services.


In [ ]:
order_process.terminate()
inventory_process.terminate()
print("Both services stopped.")

## Summary

| Topic | What we did |
|---|---|
| Data pipelines | Compared batch and streaming processing of orders |
| Microservices | Split the system into an Order Service and an Inventory Service that talk over HTTP |
| Performance trade-offs | Measured sequential vs concurrent request handling |
| Monitoring and logging | Added structured logs and a `/metrics` endpoint |
| Security | Added an API key check and input validation |

### Now we are done with week 7's lab content 🎉🎉🎉 ###
### Let's navigate to Exercise's notebook
[Exercise Notebook](Week7_Exercise.ipynb)